# 02 - Fine-Tuning IndoT5 dengan Grid Search Hyperparameter

Total eksperimen: **1 baseline + 18 grid search = 19 run**

| # | Konfigurasi | Epoch | Batch Size | Learning Rate |
|---|---|---|---|---|
| 0 | **Baseline** (default, tanpa tuning) | 3 | 8 | 2e-4 |
| 1–18 | **Grid Search** | 5, 10, 15 | 4, 8 | 1e-4, 3e-5, 5e-5 |

> Setiap run dievaluasi dengan ROUGE pada `test.csv`. Model dengan ROUGE-L tertinggi disimpan sebagai model final.

In [9]:
# Jalankan ini di cell pertama SEBELUM load model
!pip install huggingface_hub -q
from huggingface_hub import login
login()


In [2]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets evaluate sentencepiece accelerate rouge-score -q

import shutil

# ======================================================
# SESUAIKAN PATH INI dengan lokasi train.csv & test.csv
# di Google Drive kamu
DRIVE_DATA_DIR = '/content/drive/MyDrive/data_latih'  # ganti jika ada subfolder
# ======================================================

shutil.copy(f'{DRIVE_DATA_DIR}/train.csv', 'train.csv')
shutil.copy(f'{DRIVE_DATA_DIR}/test.csv',  'test.csv')
print("Data berhasil di-copy dari Google Drive!")


Mounted at /content/drive
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.9 MB/s eta 0:00:00
Data berhasil di-copy dari Google Drive!


## 1. Import & Konfigurasi

In [3]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from datasets import Dataset
import pandas as pd
import evaluate
import torch
import shutil
import itertools
from pathlib import Path

MODEL_NAME = "Wikidepia/IndoT5-base"
DATA_DIR    = Path('.')
OUTPUT_ROOT = Path('/content/drive/MyDrive/models')
BEST_DIR    = OUTPUT_ROOT / 'indot5_finetuned'
LOG_DIR     = OUTPUT_ROOT / 'training_logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

PREFIX  = 'ringkas: '
MAX_IN  = 512
MAX_OUT = 150

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device : {DEVICE}')
print(f'Model  : {MODEL_NAME}')

Device : cuda
Model  : Wikidepia/IndoT5-base


## 2. Definisi Grid & Baseline

In [4]:
# ── Baseline (hyperparameter default, tanpa tuning) ─────
BASELINE = {
    'num_train_epochs'           : 3,
    'per_device_train_batch_size': 8,
    'learning_rate'              : 2e-4,
}

# ── Grid hyperparameter sesuai tabel skripsi ─────────────
GRID = {
    'num_train_epochs'           : [5, 10, 15],
    'per_device_train_batch_size': [4, 8],
    'learning_rate'              : [1e-4, 3e-5, 5e-5],
}
keys   = list(GRID.keys())
combos = list(itertools.product(*GRID.values()))

print(f'Run 00  : BASELINE (Epoch={BASELINE["num_train_epochs"]}  '
      f'BS={BASELINE["per_device_train_batch_size"]}  LR={BASELINE["learning_rate"]})')
print(f'Run 01–{len(combos):02d}: GRID SEARCH ({len(combos)} kombinasi)')
print(f'TOTAL   : {1 + len(combos)} eksperimen')

Run 00  : BASELINE (Epoch=3  BS=8  LR=0.0002)
Run 01–18: GRID SEARCH (18 kombinasi)
TOTAL   : 19 eksperimen


## 3. Load Dataset & Tokenizer

In [5]:
df_train = pd.read_csv(DATA_DIR / 'train.csv')
df_test  = pd.read_csv(DATA_DIR / 'test.csv')
print(f'Train: {len(df_train)} | Test: {len(df_test)}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenisasi(examples):
    inputs = [PREFIX + s for s in examples['source']]
    m_in   = tokenizer(inputs, max_length=MAX_IN, truncation=True, padding='max_length')
    labels = tokenizer(
        text_target=examples['target'],
        max_length=MAX_OUT, truncation=True, padding='max_length'
    )
    m_in['labels'] = labels['input_ids']
    return m_in

train_ds = Dataset.from_pandas(df_train).map(tokenisasi, batched=True, remove_columns=df_train.columns.tolist())
test_ds  = Dataset.from_pandas(df_test).map(tokenisasi, batched=True, remove_columns=df_test.columns.tolist())
print('Tokenisasi selesai.')

Train: 24 | Test: 6


config.json:   0%|          | 0.00/696 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/777k [00:00<?, ?B/s]

Map:   0%|          | 0/24 [00:00<?, ? examples/s]

Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Tokenisasi selesai.


In [6]:
# DIAGNOSTIK: Cek apa yang dihasilkan model saat ini
model_test = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
tokenizer_test = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_input = PREFIX + df_test['source'].iloc[0][:300]
inputs = tokenizer_test(sample_input, return_tensors='pt', max_length=512, truncation=True).to(DEVICE)
outputs = model_test.generate(**inputs, max_new_tokens=100)
print("PREFIX yang dipakai:", repr(PREFIX))
print("Input awal:", sample_input[:100])
print("Output model:", tokenizer_test.decode(outputs[0], skip_special_tokens=True))
print("Referensi:", df_test['target'].iloc[0][:200])


pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


PREFIX yang dipakai: 'ringkas: '
Input awal: ringkas: Siddang Dewan yang kami hormati, ada rapat paripurna hari ini telah hadir peserta Parlement
Output model: t remajat remajat remajat remajat remaja adalaht remaja adalaht remaja adalaht remaja adalah sebuah kejutan tahunan DPRRIt remaja adalah sebuah kejutan tahunan DPRRIt remaja adalah sebuah kejutan tahunant remaja adalah sebuah kejutan tahunant remaja adalah sebuah kejutanant Remaja adalah sebuah kejutant Remaja adalah adalah: Siddang Dewan telah hadir peserta,: Siddang Dewan, Ketua,pu
Referensi: Hadirin kami persilakan untuk duduk kembali.
Sidang Dewan yang kami hormati,
Pada Rapat Paripurna hari ini, telah hadir peserta Parlemen Remaja
Tahun 2025 di atas sana, tepuk tangannya.
(TEPUK TANGAN 


## 4. Fungsi Helper

In [7]:
rouge = evaluate.load('rouge')

def hitung_rouge(model, test_dataset):
    """
    Hitung ROUGE dengan model.generate() langsung.
    Menggunakan no_repeat_ngram_size & repetition_penalty
    untuk mencegah output repetitif.
    """
    from torch.utils.data import DataLoader
    model.eval()
    preds = []
    refs  = df_test['target'].tolist()

    loader = DataLoader(test_dataset, batch_size=2)
    with torch.no_grad():
        for batch in loader:
            input_ids      = torch.tensor(batch['input_ids']).to(DEVICE)
            attention_mask = torch.tensor(batch['attention_mask']).to(DEVICE)
            outputs = model.generate(
                input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_OUT,
                no_repeat_ngram_size=3,
                repetition_penalty=2.0,
                num_beams=4,
                early_stopping=True,
            )
            decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
            preds.extend(decoded)

    preds = [p.replace(PREFIX, '').strip() for p in preds]
    if preds:
        print(f"  Sample pred[0]: {preds[0][:120]}...")
    pairs = [(p, r) for p, r in zip(preds, refs) if p.strip()]
    if not pairs:
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    p_clean, r_clean = zip(*pairs)
    scores = rouge.compute(predictions=list(p_clean), references=list(r_clean), use_stemmer=False)
    return {k: round(v, 4) for k, v in scores.items()}

def buat_trainer(model, params, run_dir):
    """Buat Seq2SeqTrainer dari dict params."""
    args = Seq2SeqTrainingArguments(
        output_dir=str(run_dir),
        num_train_epochs=params['num_train_epochs'],
        per_device_train_batch_size=params['per_device_train_batch_size'],
        per_device_eval_batch_size=4,
        learning_rate=params['learning_rate'],
        warmup_steps=50,
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy='epoch',
        save_strategy='no',
        predict_with_generate=True,
        generation_max_length=MAX_OUT,
        fp16=(DEVICE == 'cuda'),
        report_to='none',
    )
    return Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=test_ds,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
        processing_class=tokenizer,
    )

def jalankan_satu_run(run_label, params, run_dir, hasil_list, best_state):
    """Training + evaluasi 1 run, update best model jika perlu."""
    print(f'\n{"="*55}')
    print(f'[{run_label}] Epoch={params["num_train_epochs"]}  '
          f'BS={params["per_device_train_batch_size"]}  '
          f'LR={params["learning_rate"]}')
    print(f'{"="*55}')

    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
    trainer = buat_trainer(model, params, run_dir)
    trainer.train()

    scores = hitung_rouge(model, test_ds)
    print(f'  ROUGE-1={scores["rouge1"]}  ROUGE-2={scores["rouge2"]}  ROUGE-L={scores["rougeL"]}')

    hasil_list.append({
        'run'          : run_label,
        'epoch'        : params['num_train_epochs'],
        'batch_size'   : params['per_device_train_batch_size'],
        'learning_rate': params['learning_rate'],
        'rouge1'       : scores['rouge1'],
        'rouge2'       : scores['rouge2'],
        'rougeL'       : scores['rougeL'],
    })

    if scores['rougeL'] > best_state['rouge_l']:
        best_state['rouge_l'] = scores['rougeL']
        best_state['config']  = params.copy()
        best_state['dir']     = run_dir
        trainer.save_model(str(run_dir))
        tokenizer.save_pretrained(str(run_dir))
        print(f'  ⭐ Model terbaik baru! ROUGE-L={scores["rougeL"]}')

    del model, trainer
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


## 5. Run 00 — Baseline (Default Hyperparameter)

In [ ]:
hasil_eksperimen = []
best_state = {'rouge_l': -1, 'config': None, 'dir': None}

jalankan_satu_run(
    run_label='00_baseline',
    params=BASELINE,
    run_dir=OUTPUT_ROOT / 'run_00_baseline',
    hasil_list=hasil_eksperimen,
    best_state=best_state,
)


[00_baseline] Epoch=3  BS=8  LR=0.0002


Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Epoch,Training Loss,Validation Loss


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.18 GiB is allocated by PyTorch, and 237.34 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 6. Run 01–18 — Grid Search Hyperparameter

In [ ]:
for run_idx, combo in enumerate(combos, start=1):
    params   = dict(zip(keys, combo))
    run_name = f'{run_idx:02d}_ep{params["num_train_epochs"]}_bs{params["per_device_train_batch_size"]}_lr{params["learning_rate"]}'
    jalankan_satu_run(
        run_label=run_name,
        params=params,
        run_dir=OUTPUT_ROOT / f'run_{run_name}',
        hasil_list=hasil_eksperimen,
        best_state=best_state,
    )

print('\n✅ Semua 19 eksperimen selesai!')

## 7. Ringkasan & Visualisasi Hasil

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

df_hasil = pd.DataFrame(hasil_eksperimen).sort_values('rougeL', ascending=False).reset_index(drop=True)
print('\n📊 SEMUA HASIL (urut ROUGE-L tertinggi):')
print(df_hasil.to_string(index=False))
df_hasil.to_csv(OUTPUT_ROOT / 'hasil_grid_search.csv', index=False)

# ── Grafik 1: ROUGE-L keseluruhan ────────────────────────
fig, ax = plt.subplots(figsize=(14, 4))
colors = ['#E74C3C' if r == '00_baseline' else '#4A90D9' for r in df_hasil['run']]
ax.bar(df_hasil['run'], df_hasil['rougeL'], color=colors)
ax.set_title('ROUGE-L per Eksperimen (merah = baseline)', fontsize=12)
ax.set_xlabel('Run')
ax.set_ylabel('ROUGE-L')
plt.xticks(rotation=45, ha='right', fontsize=7)
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'rouge_per_run.png', dpi=150)
plt.show()

# ── Grafik 2: Rata-rata per hyperparameter ────────────────
df_grid = df_hasil[df_hasil['run'] != '00_baseline']
fig2, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
fig2.suptitle('Pengaruh Hyperparameter terhadap ROUGE-L (grid only)', fontsize=12)
for ax, col, label in zip(axes,
    ['epoch', 'batch_size', 'learning_rate'],
    ['Epoch', 'Batch Size', 'Learning Rate']):
    g = df_grid.groupby(col)['rougeL'].mean().reset_index()
    ax.bar(g[col].astype(str), g['rougeL'], color='#2ECC71')
    ax.set_title(f'ROUGE-L vs {label}')
    ax.set_xlabel(label)
    ax.set_ylabel('Rata-rata ROUGE-L')
plt.tight_layout()
plt.savefig(OUTPUT_ROOT / 'visualisasi_grid_search.png', dpi=150)
plt.show()
print('Grafik disimpan.')

## 8. Simpan Model Terbaik → `models/indot5_finetuned/`

In [ ]:
print(f'\n🏆 Model terbaik:')
for k, v in best_state['config'].items():
    print(f'   {k}: {v}')
print(f'   ROUGE-L: {best_state["rouge_l"]}')

if BEST_DIR.exists():
    shutil.rmtree(BEST_DIR)
shutil.copytree(str(best_state['dir']), str(BEST_DIR))
print(f'\n✅ Model disalin ke: {BEST_DIR}')
print('   → Siap digunakan oleh app.py')